# Customer Churn Prediction — Exploratory Data Analysis
**Simrah Ayan** | Durham College — Data Analytics for Business Decision Making

Dataset: IBM Telco Customer Churn (via Kaggle)

Objective: Understand what factors drive customer churn, then prepare data for ML modeling.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_palette('Set2')

print('Libraries loaded.')

In [ ]:
df = pd.read_csv('data/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Shape: {df.shape}')
df.head()

## 1. Basic Overview

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Missing Values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])
print('\n=== Churn Distribution ===')
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).round(3) * 100, '%')

## 2. Churn Rate Overview

In [ ]:
fig, ax = plt.subplots()
counts = df['Churn'].value_counts()
colors = ['#1D9E75', '#E24B4A']
ax.bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
ax.set_title('Customer Churn Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Customers')
for i, v in enumerate(counts.values):
    ax.text(i, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('plots/01_churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Churn by Contract Type

In [ ]:
churn_by_contract = df.groupby('Contract')['Churn'].apply(lambda x: (x == 'Yes').mean() * 100).reset_index()
churn_by_contract.columns = ['Contract', 'Churn Rate (%)']
fig, ax = plt.subplots()
ax.bar(churn_by_contract['Contract'], churn_by_contract['Churn Rate (%)'],
       color=['#1D9E75', '#BA7517', '#E24B4A'], edgecolor='white')
ax.set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
for i, row in churn_by_contract.iterrows():
    ax.text(i, row['Churn Rate (%)'] + 0.5, f"{row['Churn Rate (%)']:.1f}%", ha='center')
plt.tight_layout()
plt.savefig('plots/02_churn_by_contract.png', dpi=150, bbox_inches='tight')
plt.show()
print(churn_by_contract)

## 4. Monthly Charges vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2)
for ax, col in zip(axes, ['MonthlyCharges', 'tenure']):
    churned = df[df['Churn'] == 'Yes'][col]
    stayed = df[df['Churn'] == 'No'][col]
    ax.hist(stayed, bins=30, alpha=0.6, label='No Churn', color='#1D9E75')
    ax.hist(churned, bins=30, alpha=0.6, label='Churned', color='#E24B4A')
    ax.set_title(f'{col} Distribution', fontweight='bold')
    ax.set_xlabel(col)
    ax.legend()
plt.tight_layout()
plt.savefig('plots/03_charges_tenure.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap (Numeric Features)

In [ ]:
df_num = df[['tenure', 'MonthlyCharges', 'TotalCharges']].copy()
df_num['TotalCharges'] = pd.to_numeric(df_num['TotalCharges'], errors='coerce')
df_num['Churn'] = (df['Churn'] == 'Yes').astype(int)
fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(df_num.corr(), annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.savefig('plots/04_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Key EDA Findings

- **Churn rate is ~26.5%** — class imbalance to handle in modeling.
- **Month-to-month contracts** have dramatically higher churn (~43%) vs 2-year contracts (~3%).
- **Higher monthly charges** correlate with higher churn.
- **Shorter tenure** strongly associated with churning — early retention is critical.
- **TotalCharges** is highly correlated with tenure (as expected).